# 01 - Data Loading, Schema & Unit of Analysis



## Setup




In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_PATH = Path('../../data/raw data/DataCoSupplyChainDataset.csv')  

try:
    df = pd.read_csv(DATA_PATH, encoding='utf-8')
except UnicodeDecodeError:
    print("UTF-8 failed, falling back to ISO-8859-1 (common for this dataset)")
    df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1')

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


UTF-8 failed, falling back to ISO-8859-1 (common for this dataset)
Shape: 180,519 rows x 53 columns


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


## 1. Schema overview



In [7]:
schema_overview = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'n_unique': [df[c].nunique() for c in df.columns],
    'n_missing': df.isnull().sum().values,
    'pct_missing': (df.isnull().sum().values / len(df) * 100).round(2),
    'sample_value': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})
schema_overview.sort_values('pct_missing', ascending=False)


,column,dtype,n_unique,n_missing,pct_missing,sample_value
46,Product Description,float64,0,180519,100.00,None
43,Order Zipcode,float64,609,155679,86.24,99301.0
2,Days for shipment (scheduled),int64,4,0,0.00,4
1,Days for shipping (real),int64,7,0,0.00,3
0,Type,object,4,0,0.00,DEBIT
5,Delivery Status,object,4,0,0.00,Advance shipping
6,Late_delivery_risk,int64,2,0,0.00,0
7,Category Id,int64,51,0,0.00,73
8,Category Name,object,50,0,0.00,Sporting Goods
9,Customer City,object,563,0,0.00,Caguas


**Order time safe fields (candidates):** `Type`, `Days for shipment (scheduled)`, `Shipping Mode`, `Order Region`, `Order Country`, `Order City`, `Order State`, `Category Id`, `Category Name`, `Order Item Quantity`, `Order Item Product Price`, `Sales`, `Customer Segment`, `order date (DateOrders)`.

**Post outcome fields - exclude from model inputs:** `Delivery Status`, `Late_delivery_risk` (target), `Days for shipping (real)`, `shipping date (DateOrders)`.

**What we found:**

- **Dataset confirmed at 180,519 rows × 53 columns** - matches the expected size for this dataset, so we're working with the correct/complete file.
- **`Product Description` is 100% missing** (all 180,519 rows). This column carries zero information and should simply be **dropped** in Stage 3 - not imputed, since there's nothing to impute from.
- **`Order Zipcode` is 86.24% missing.** This is a large gap, but it looks **systemic rather than random** - it's very likely only populated for specific countries (e.g. US style zip codes don't apply the same way globally). We should check in Stage 3 whether missingness correlates with `Order Country` before deciding whether to drop the column entirely or keep it as a "zipcode known / not known" indicator.
- **`Customer Lname` (8 missing) and `Customer Zipcode` (3 missing)** are negligible - safe to drop those few rows or impute trivially, not worth a complex strategy.
- **Every other column has 0% missing values** - this dataset is otherwise very clean, which is good news for Stage 3 (minimal imputation work needed overall).
- **Interesting/important data quirk to flag in our report:** `Customer Country` only has **2 unique values** (Puerto Rico and the US, shown as "EE. UU."), while `Order Country` has **164 unique values** and `Order City` has 3,597. This means the dataset's *customer master records* are synthetic/limited to just two countries, even though *orders* ship globally. This is a known characteristic of this generated dataset, not a data error on our part - worth one sentence in our data quality write up so it doesn't look like we missed it, and it means we should rely on `Order Country`/`Order Region`/`Order City` (not `Customer Country`) for any geographic feature engineering, since that's where the real geographic variation lives.
- **Minor oddity worth a quick check:** `Category Id` has 51 unique values but `Category Name` has only 50 - meaning either two IDs share one name, or one ID has an inconsistent/missing in practice name mapping. Small thing, but worth a one line check in Stage 3 before we rely on Category Name for feature engineering, to make sure we're not silently collapsing two distinct categories into one.
- **`order date (DateOrders)` is currently stored as text (`object` dtype)**, not a real datetime - this confirms we'll need to explicitly parse it with `pd.to_datetime()` before any temporal feature engineering (which the geographic/temporal notebook already does).


## 2. Unit of analysis

Is the data one row per order, or per order line?

In [4]:
n_rows = len(df)
n_unique_orders = df['Order Id'].nunique()
n_unique_order_items = df['Order Item Id'].nunique() if 'Order Item Id' in df.columns else None

print(f"Total rows: {n_rows:,}")
print(f"Unique Order Id values: {n_unique_orders:,}")
if n_unique_order_items:
    print(f"Unique Order Item Id values: {n_unique_order_items:,}")

if n_rows == n_unique_orders:
    print("\n--> Data is at ORDER level (one row per order).")
else:
    avg_lines_per_order = n_rows / n_unique_orders
    print(f"\n--> Data is at ORDER-LINE level. Average {avg_lines_per_order:.2f} lines per order.")


Total rows: 180,519
Unique Order Id values: 65,752
Unique Order Item Id values: 180,519

--> Data is at ORDER-LINE level. Average 2.75 lines per order.


In [5]:
if n_rows != n_unique_orders:
    sample_order_id = df['Order Id'].value_counts().index[0]
    display(df[df['Order Id'] == sample_order_id])


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
90419,PAYMENT,6,4,43.220001,123.489998,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,XXXXXXXXX,Jessica,9903,Smith,XXXXXXXXX,Corporate,PR,8663 Crystal Bluff Walk,725.0,4,Apparel,18.253716,-66.370522,Africa,Rabat,Marruecos,9903,1/4/2017 2:28,50290,403,6.500000,0.05,125677,129.990005,0.35,1,129.990005,123.489998,43.220001,North Africa,Rabat-Salé-Zemur-Zaer,PENDING_PAYMENT,NaN,403,18,NaN,http://images.acmesports.sports/Nike+Men%27s+C...,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,0,1/10/2017 2:28,Standard Class
90420,PAYMENT,6,4,29.240000,122.839996,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,XXXXXXXXX,Jessica,9903,Smith,XXXXXXXXX,Corporate,PR,8663 Crystal Bluff Walk,725.0,4,Apparel,18.253716,-66.370522,Africa,Rabat,Marruecos,9903,1/4/2017 2:28,50290,403,7.150000,0.06,125676,129.990005,0.24,1,129.990005,122.839996,29.240000,North Africa,Rabat-Salé-Zemur-Zaer,PENDING_PAYMENT,NaN,403,18,NaN,http://images.acmesports.sports/Nike+Men%27s+C...,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,0,1/10/2017 2:28,Standard Class
90422,PAYMENT,6,4,-88.610001,120.889999,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,XXXXXXXXX,Jessica,9903,Smith,XXXXXXXXX,Corporate,PR,8663 Crystal Bluff Walk,725.0,4,Apparel,18.253716,-66.370522,Africa,Rabat,Marruecos,9903,1/4/2017 2:28,50290,403,9.100000,0.07,125675,129.990005,-0.73,1,129.990005,120.889999,-88.610001,North Africa,Rabat-Salé-Zemur-Zaer,PENDING_PAYMENT,NaN,403,18,NaN,http://images.acmesports.sports/Nike+Men%27s+C...,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,0,1/10/2017 2:28,Standard Class
91680,PAYMENT,6,4,78.610001,163.770004,Late delivery,1,17,Cleats,Caguas,Puerto Rico,XXXXXXXXX,Jessica,9903,Smith,XXXXXXXXX,Corporate,PR,8663 Crystal Bluff Walk,725.0,4,Apparel,18.253716,-66.370522,Africa,Rabat,Marruecos,9903,1/4/2017 2:28,50290,365,16.200001,0.09,125678,59.990002,0.48,3,179.970001,163.770004,78.610001,North Africa,Rabat-Salé-Zemur-Zaer,PENDING_PAYMENT,NaN,365,17,NaN,http://images.acmesports.sports/Perfect+Fitnes...,Perfect Fitness Perfect Rip Deck,59.990002,0,1/10/2017 2:28,Standard Class
111298,PAYMENT,6,4,37.040001,142.440002,Late delivery,1,46,Indoor/Outdoor Games,Caguas,Puerto Rico,XXXXXXXXX,Jessica,9903,Smith,XXXXXXXXX,Corporate,PR,8663 Crystal Bluff Walk,725.0,7,Fan Shop,18.253716,-66.370522,Africa,Rabat,Marruecos,9903,1/4/2017 2:28,50290,1014,7.500000,0.05,125674,49.980000,0.26,3,149.940002,142.440002,37.040001,North Africa,Rabat-Salé-Zemur-Zaer,PENDING_PAYMENT,NaN,1014,46,NaN,http://images.acmesports.sports/O%27Brien+Men%...,O'Brien Men's Neoprene Life Vest,49.980000,0,1/10/2017 2:28,Standard Class


**What we found:**

- **The dataset is confirmed to be at ORDER LINE level, not order level.** 180,519 total rows collapse to only **65,752 unique orders** - an average of **2.75 line items per order**.
- **This directly determines our first Stage 3 preprocessing step**: we must aggregate line items up to a single row per `Order Id` before we can treat "will this order be late" as a clean, one prediction per order classification problem. Right now the same order shows up 2-3 times in the raw data, which would badly bias any model trained on it as is (the model would effectively see the same outcome repeated, inflating its apparent confidence).
- **Good news from the sample order we inspected (Order Id 50290, 5 line items):** `Delivery Status` and `Late_delivery_risk` are **identical across every line item within the same order** (all showing "Late delivery" / 1). This confirms delivery outcome is genuinely an **order level property**, not a line item level one in this dataset - so our target variable aggregates cleanly with a simple rule like `.groupby('Order Id')['Late_delivery_risk'].first()` (or `.max()` as a safety net if any order ever showed mixed values, which we should double check for, but this sample suggests it won't happen).
- **Numeric fields will need a real aggregation decision, not just `.first()`.** Fields like `Sales`, `Order Item Quantity`, `Benefit per order`, and `Order Item Product Price` vary *across* line items within the same order (see the sample: prices range from $49.98 to $129.99 within order 50290). For these, Stage 3 needs to decide per field: **sum** (e.g. total `Sales` per order - makes sense for our secondary lens's order value calculation), **mean**, or **count** (e.g. number of line items as a new engineered feature - could itself be a useful predictor, since larger/more complex orders might have different delay risk).
- **Practical next step:** log this aggregation decision explicitly in `DECISION_LOG.md` before Stage 3 begins, specifying the aggregation rule per column (sum for `Sales`/`Order Item Quantity`, first/mode for categorical order-level fields like `Shipping Mode` and `Order Region`, first for the target). This is exactly the kind of "options considered, reasoning" entry that scores well - and it directly resolves the "unit of analysis" open question from our original project framing.
